# Capítulo 5 · Actividad: ¿quién sobrevivió al Titanic?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/01_actividad_titanic_arbol.ipynb)

Entrena un **árbol de decisión** que prediga si un pasajero sobrevivió (`survived`) y consigue el mejor modelo que puedas. Haz todo el proceso que vimos en clase; el preprocesamiento lo decides tú.

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`

In [1]:
from pathlib import Path

import pandas as pd

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

df = pd.read_csv(DATA_DIR / "titanic.csv")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
# Tu código aquí
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer

# 1) Limpieza de filas (no aprende nada, va antes del split)
df = titanic.drop(columns=["embark_town", "adult_male", "class", "alive", "alone", "deck"]).drop_duplicates()
errores = (df["age"] < 1) | (df["fare"] > 500) | (df["fare"] <= 0) | (df["sibsp"] >= 8)
df = df[~errores]

X = df.drop(columns="survived")
y = df["survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 2) Imputar age con la mediana por (sex, who), aprendida solo con train
class ImputadorEdad(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.medianas_ = X.groupby(["sex", "who"])["age"].median()
        self.mediana_global_ = X["age"].median()
        return self

    def transform(self, X):
        X = X.copy()
        por_grupo = pd.MultiIndex.from_frame(X[["sex", "who"]]).map(self.medianas_)
        X["age"] = X["age"].fillna(pd.Series(por_grupo, index=X.index).fillna(self.mediana_global_))
        return X

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_


# 3) Features nuevas
def crear_features(X):
    X = X.copy()
    X["fsize"] = X["sibsp"] + X["parch"]
    X["u_fare"] = X["fare"] / (X["fsize"] + 1)
    X["r_age"] = pd.cut(X["age"], bins=[0, 15, 50, np.inf], labels=["child", "adult", "eldery"]).astype(str)
    return X

def nombres_features(transformer, nombres):
    return [*nombres, "fsize", "u_fare", "r_age"]


# 4) Transformación
pipeline_asimetrico = Pipeline([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("escalar", StandardScaler()),
])

columnas = ColumnTransformer([
    ("num", StandardScaler(), ["pclass", "age", "sibsp", "parch", "fsize"]),
    ("asim", pipeline_asimetrico, ["fare", "u_fare"]),
    ("bin", OneHotEncoder(drop="if_binary", sparse_output=False), ["sex"]),
    ("nom", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), ["embarked", "who"]),
    ("ord", OrdinalEncoder(categories=[["child", "adult", "eldery"]]), ["r_age"]),
], verbose_feature_names_out=False)

pipe = Pipeline([
    ("imputar_embarked", ColumnTransformer(
        [("moda", SimpleImputer(strategy="most_frequent"), ["embarked"])],
        remainder="passthrough", verbose_feature_names_out=False)),
    ("imputar_edad", ImputadorEdad()),
    ("features", FunctionTransformer(crear_features, feature_names_out=nombres_features)),
    ("columnas", columnas),
])

X_train_t = pipe.fit_transform(X_train)
X_test_t = pipe.transform(X_test)
